# A1 — JupyterHub A100, ưu tiên hạn thứ Hai 09/11/2026
Không cần Colab, mount Drive, terminal UI hoặc Gradio. Chạy trong kernel GPU công ty. Repo phải được push từ máy cá nhân trước khi clone/pull. Không tự động upload dữ liệu ra ngoài công ty.

Hàng đợi hoàn tất đủ R1–R8 seed 42 trước, rồi các seed pretrained 43/44. Giữ nguyên epoch theo A1.1; không tự rút ngắn để vừa thời gian. Không chạy A2 cùng lúc trên một GPU khi A1 còn thiếu.


## 1. Repo và kernel GPU

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
REPO = Path.home()/'SDH_HK261_CO5085'  # Sửa nếu repo được clone chỗ khác.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/truong1501/SDH_HK261_CO5085.git',str(REPO)],check=True)
assert (REPO/'A1/src/server.py').is_file(), 'Cần clone/pull bản code mới đã push lên Git'
import torch
print('Python:',sys.version)
assert torch.cuda.is_available(), 'Chọn kernel GPU của công ty; không tự thay CUDA hệ thống'
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/2**30)


## 2. Phụ thuộc và tests
Sử dụng PyTorch/torchvision tương thích GPU được server cung cấp. pip không có --upgrade; log hiển thị đầy đủ. Tests không tải pretrained hoặc Food-101.

In [ ]:
os.chdir(REPO/'A1')
result=subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(result.stdout);result.check_returncode()
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)


## 3. Dữ liệu, run đã có và checkpoint resume
- Đặt `DATA_ROOT` trên ổ local/scratch bền vững theo quy định công ty; cần đủ dung lượng Food-101.
- Nếu có archive Food-101, upload qua JupyterHub hoặc cơ chế công ty cho phép rồi đặt `FOOD_ARCHIVE`; nếu không, runner tải từ nguồn torchvision/ETH.
- Clone Git mang metrics/config/history nếu đã commit, nhưng **không** mang các file .pt/cache đang được gitignore.
- Để resume R3_s42, chuyển `last.pt`, `best.pt`, `config.yaml`, `history.csv` vào `OUT/R3_s42/`. Không có checkpoint thì runner train lại riêng run còn dở, giữ nguyên các run có metrics.
- Giữ thư mục output trên ổ không bị xóa khi phiên 12 giờ kết thúc. Không cần chuyển cache feature: path khác sẽ tạo cache mới.


In [ ]:
DATA_ROOT = Path.home()/'datasets'  # Nên đổi sang /scratch/... nếu có.
OUT = REPO/'A1/runs'
SPLITS = REPO/'A1/splits'
FOOD_ARCHIVE = None  # Ví dụ Path('/scratch/food-101.tar.gz')
DATA_ROOT.mkdir(parents=True,exist_ok=True)
if FOOD_ARCHIVE is not None and not (DATA_ROOT/'food-101/images').is_dir():
    subprocess.run(['tar','-xzf',str(FOOD_ARCHIVE),'-C',str(DATA_ROOT)],check=True)
subprocess.run([sys.executable,'-m','src.server','--data_root',str(DATA_ROOT),'--out_root',str(OUT),
                '--splits_dir',str(SPLITS),'--dry_run'],check=True)


## 4. Chạy hàng đợi A1
Mặc định 11 giờ, chừa khoảng một giờ cho export trước giới hạn 12 giờ. Đây là **giới hạn mềm tại ranh giới epoch**: không cắt tiến trình giữa lúc ghi checkpoint. Thời gian tải dữ liệu/epoch bất thường có thể vượt dự báo; không hứa hoàn tất tất cả run trong một phiên. Runner bỏ qua metrics đã có, resume standard từ last.pt. Nếu công ty dừng session đột ngột, chỉ epoch đã lưu hoàn chỉnh được bảo toàn.


In [ ]:
HOURS = 11
command=[sys.executable,'-u','-m','src.server','--data_root',str(DATA_ROOT),
         '--out_root',str(OUT),'--splits_dir',str(SPLITS),'--hours',str(HOURS),'--num_workers','4']
OUT.mkdir(parents=True,exist_ok=True)
log_path=OUT/'jupyterhub_session.log'
with log_path.open('a',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='');log.write(line);log.flush()
    returncode=process.wait()
if returncode not in (0,75):
    raise RuntimeError(f'A1 lỗi ({returncode}); xem {log_path}. Không coi là hoàn tất.')
print((OUT/'session_status.json').read_text())


## 5. Tổng hợp kết quả đã hoàn tất
Bảng chỉ gồm metrics thật, loại smoke và run dò LR. Nếu chưa đủ 20 run, đây là báo cáo tiến độ; không khẳng định hoàn tất protocol.

In [ ]:
subprocess.run([sys.executable,'-m','src.summarize','--out_root',str(OUT),
                '--splits_dir',str(SPLITS),'--fig_dir','figures'],check=True)
from IPython.display import display,Markdown,Image,FileLink
display(Markdown(Path('summary.md').read_text(encoding='utf-8')))
for name in ['curves','tradeoff','pretrain_gap']:
    path=Path('figures')/f'{name}.png'
    if path.exists():display(Image(filename=str(path)))


## 6. Xuất kết quả để tải về / đưa lên Drive
Archive nhỏ không chứa ảnh Food-101, .pt hoặc cache feature. FileLink là link tải trong JupyterHub; bạn tự chuyển file về máy/Drive theo quy định công ty. Nếu cần resume ở máy khác, tạo archive checkpoint riêng bằng cell tùy chọn phía sau.


In [ ]:
import zipfile
EXPORT=REPO/'A1/exports';EXPORT.mkdir(exist_ok=True)
archive=EXPORT/'A1_results.zip'
files=[]
for folder in [OUT,SPLITS,REPO/'A1/figures']:
    files += [p for p in folder.rglob('*') if p.is_file() and p.suffix in {'.json','.yaml','.csv','.npy','.npz','.png'}
              and '_smoke' not in str(p) and '_features' not in str(p)]
files += [p for p in [Path('summary.md'),Path('summary.csv'),Path('runs_all.csv')] if p.exists()]
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in files:z.write(p,p.resolve().relative_to((REPO/'A1').resolve()))
print('Archive:',archive,'MB',archive.stat().st_size/2**20)
display(FileLink(str(archive.relative_to(Path.cwd())), url_prefix='../'))


## 7. Tùy chọn: đóng gói checkpoint các run chưa xong
Chỉ chạy nếu cần chuyển máy. Có thể lớn; không commit Git. Kiểm tra dung lượng đĩa trước khi chạy.

In [ ]:
EXPORT_CHECKPOINTS = False
if EXPORT_CHECKPOINTS:
    checkpoint_archive=EXPORT/'A1_resume_checkpoints.zip'
    with zipfile.ZipFile(checkpoint_archive,'w',compression=zipfile.ZIP_STORED) as z:
        for directory in OUT.glob('R*_s*'):
            if directory.name.endswith('_smoke') or (directory/'metrics.json').exists():continue
            for name in ['last.pt','best.pt','config.yaml','history.csv']:
                p=directory/name
                if p.exists():z.write(p,p.relative_to(REPO/'A1'))
    display(FileLink(str(checkpoint_archive.relative_to(Path.cwd())), url_prefix='../'))
